# LSTM & GRU

**Companion lesson:** https://ml-viz.vercel.app/courses/rnns/03-lstm-and-gru

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## An LSTM cell, fully from scratch

Three gates and an **additive** cell-state update — the gradient highway that fixes the vanishing gradient from the previous lesson.

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))

class LSTMCell:
    def __init__(self, n_in, nh):
        k = n_in + nh
        self.nh = nh
        self.Wf=np.random.randn(nh,k)*0.1; self.bf=np.ones((nh,1))   # forget bias=1
        self.Wi=np.random.randn(nh,k)*0.1; self.bi=np.zeros((nh,1))
        self.Wc=np.random.randn(nh,k)*0.1; self.bc=np.zeros((nh,1))
        self.Wo=np.random.randn(nh,k)*0.1; self.bo=np.zeros((nh,1))

    def step(self, x, h, c):
        z = np.vstack([h, x])
        f = sigmoid(self.Wf@z + self.bf)
        i = sigmoid(self.Wi@z + self.bi)
        g = np.tanh(self.Wc@z + self.bc)
        o = sigmoid(self.Wo@z + self.bo)
        c = f*c + i*g                 # additive update = gradient highway
        h = o*np.tanh(c)
        return h, c, dict(f=f, i=i, o=o)

cell = LSTMCell(n_in=3, nh=5)
h = c = np.zeros((5,1))
for t in range(4):
    h, c, gates = cell.step(np.random.randn(3,1), h, c)
    print(f't={t}  mean forget gate={gates["f"].mean():.2f}  ||cell||={np.linalg.norm(c):.2f}')

## A GRU cell, from scratch

Two gates (reset, update), one state vector — the update gate does the forget+input job.

In [ ]:
class GRUCell:
    def __init__(self, n_in, nh):
        k = n_in + nh; self.nh = nh
        self.Wz=np.random.randn(nh,k)*0.1; self.Wr=np.random.randn(nh,k)*0.1
        self.Wh=np.random.randn(nh,k)*0.1

    def step(self, x, h):
        z = sigmoid(self.Wz@np.vstack([h,x]))
        r = sigmoid(self.Wr@np.vstack([h,x]))
        hh = np.tanh(self.Wh@np.vstack([r*h, x]))
        return (1-z)*h + z*hh

gru = GRUCell(3, 5); h = np.zeros((5,1))
for t in range(4):
    h = gru.step(np.random.randn(3,1), h)
print('GRU final hidden:', np.round(h.ravel(), 3))

## The whole point: memory survives long gaps

Compare how a signal injected at $t=0$ persists in an LSTM cell state vs a vanilla RNN hidden state, when the forget gate stays open (≈1). The LSTM keeps it; the RNN washes it out through repeated `tanh` squashing.

In [ ]:
T = 60
# LSTM with forget gate held open and no new input: c_t = c_0
c = np.ones((1,1)); lstm_mem = []
for t in range(T):
    f = sigmoid(np.array([[3.0]]))     # forget gate ~0.95 (bias open)
    c = f*c + 0.0                       # no new input written
    lstm_mem.append(c.item())
# Vanilla RNN with small recurrent weight, signal decays
h = 1.0; rnn_mem = []
for t in range(T):
    h = np.tanh(0.9*h)
    rnn_mem.append(h)
plt.plot(lstm_mem, label='LSTM cell state (gate open)', color='#14b8a6')
plt.plot(rnn_mem, label='vanilla RNN hidden state', color='#f43f5e')
plt.xlabel('time step'); plt.ylabel('retained signal'); plt.legend()
plt.title('Long-term memory: LSTM cell vs vanilla RNN'); plt.show()

## Key takeaways

- The LSTM's additive cell update lets memory (and gradients) survive long gaps.
- Three gates (forget/input/output) learn what to keep, write, and expose.
- The GRU achieves the same with two gates and one state — fewer parameters.
- Opening the forget gate preserves a signal indefinitely, unlike a vanilla RNN.